# NACC Baseline Modeling (starter)
This notebook trains a simple baseline model on `output/cohort.csv`.

- Model: multinomial logistic regression (balanced)
- Preprocessing: impute + one-hot for categoricals
- Metrics: macro-F1 + per-class report


In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

OUT_DIR  = os.path.join("..", "output")
COHORT_PATH = os.path.join(OUT_DIR, "cohort.csv")

df = pd.read_csv(COHORT_PATH)
print("Loaded:", df.shape)
df["label"].value_counts(dropna=False)


In [ ]:
# Keep only main 4-class cohort for primary analysis
keep = ["AD", "Vascular", "PD_LBD", "NoDementia_5y"]
df = df[df["label"].isin(keep)].copy()
print("Filtered:", df.shape)
df["label"].value_counts()


In [ ]:
y = df["label"].copy()
X = df.drop(columns=["label"], errors="ignore")

# Drop obvious identifiers / dates
X = X.drop(columns=[c for c in ["NACCID","baseline_date","dementia_date","etiology_class"] if c in X.columns], errors="ignore")

cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)

model = LogisticRegression(
    max_iter=4000,
    class_weight="balanced",
    multi_class="multinomial"
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = Pipeline([("prep", preprocess), ("model", model)])
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
print(classification_report(y_test, pred))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, pred, labels=keep))
